# NASA C-MAPSS Turbofan Engine Degradation Data

Loads the 4 sub-datasets (FD001-FD004) from `archive/`. Each has:
- `train_FD00x.txt`: run-to-failure training trajectories
- `test_FD00x.txt`: truncated test trajectories
- `RUL_FD00x.txt`: true remaining-useful-life for each test unit

Files are whitespace-separated, no header, 26 columns.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("archive")

# Column names per the readme: unit id, cycle, 3 op settings, 21 sensors
COLUMNS = (
    ["unit", "cycle", "op_setting_1", "op_setting_2", "op_setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)


def load_cmapss_file(path):
    return pd.read_csv(path, sep=r"\s+", header=None, names=COLUMNS)


def load_rul_file(path):
    return pd.read_csv(path, sep=r"\s+", header=None, names=["RUL"])

In [ ]:
datasets = {}

for fd in ["FD001", "FD002", "FD003", "FD004"]:
    datasets[fd] = {
        "train": load_cmapss_file(DATA_DIR / f"train_{fd}.txt"),
        "test": load_cmapss_file(DATA_DIR / f"test_{fd}.txt"),
        "rul": load_rul_file(DATA_DIR / f"RUL_{fd}.txt"),
    }

for fd, d in datasets.items():
    print(fd, "train:", d["train"].shape, "test:", d["test"].shape, "rul:", d["rul"].shape)

In [ ]:
datasets["FD001"]["train"].head()

## Adding RUL to the training set

Training files run each engine to failure, so RUL at each row = (max cycle for that unit) - (current cycle). The `RUL_FD00x.txt` files only apply to the *test* set (they give the true remaining life at the point each test trajectory was truncated).

In [ ]:
def add_train_rul(df):
    max_cycle = df.groupby("unit")["cycle"].transform("max")
    df = df.copy()
    df["RUL"] = max_cycle - df["cycle"]
    return df


for fd, d in datasets.items():
    d["train"] = add_train_rul(d["train"])

datasets["FD001"]["train"].tail()